In [4]:
# ═══════════════════════════════════════════════════════════════════
#  REPO SIZER — counts tokens, lines, and files across a codebase
# ═══════════════════════════════════════════════════════════════════

import tiktoken
import pandas as pd
from pathlib import Path

# ─── Configuration ─────────────────────────────────────────────────
REPO_PATH = REPO_PATH = "/Users/robertandersson/dev/satellite-worker"  # "." = current dir; or absolute path like "/Users/you/code/myrepo"

extensions = {
    ".ts", ".tsx", ".js", ".jsx", ".mjs",   # JS/TS
    ".html", ".css",                         # web
    ".json", ".md", ".yaml", ".yml",  ".md",       # config + docs
    ".py",                                   # python (if any)
}

ignore_dirs = {
    "node_modules", ".git", "dist", "build",
    ".wrangler", ".next", ".vscode", "coverage",
    ".venv", "venv", "__pycache__",
}

# Files that bloat counts but aren't really "code you maintain"
ignore_files = {
    "package-lock.json", "yarn.lock", "pnpm-lock.yaml",
}

# ─── Tokenizer setup ───────────────────────────────────────────────
enc = tiktoken.get_encoding("cl100k_base")

# ─── Walk the repo ─────────────────────────────────────────────────
rows = []
skipped = 0

for path in Path(REPO_PATH).rglob("*"):
    if not path.is_file():
        continue
    if path.suffix not in extensions:
        continue
    if path.name in ignore_files:
        continue
    if any(part in ignore_dirs for part in path.parts):
        continue
    try:
        content = path.read_text(encoding="utf-8")
        rows.append({
            "path": str(path),
            "ext": path.suffix,
            "tokens": len(enc.encode(content)),
            "lines": content.count("\n") + 1,
            "bytes": len(content.encode("utf-8")),
        })
    except (UnicodeDecodeError, PermissionError):
        skipped += 1
        continue

df = pd.DataFrame(rows)

if df.empty:
    print("No matching files found. Check REPO_PATH and the extensions set.")
else:
    total_tokens = df["tokens"].sum()
    total_lines = df["lines"].sum()
    total_files = len(df)

    # ─── Headline ──────────────────────────────────────────────────
    print("═" * 64)
    print("  REPO SIZE SUMMARY")
    print("═" * 64)
    print(f"  Path:   {Path(REPO_PATH).resolve()}")
    print(f"  Files:  {total_files:>10,}")
    print(f"  Lines:  {total_lines:>10,}")
    print(f"  Tokens: {total_tokens:>10,}  (cl100k_base, ~equivalent to Claude)")
    if skipped:
        print(f"  Skipped (binary/permission): {skipped}")
    print()

    # ─── Verdict ───────────────────────────────────────────────────
    if total_tokens < 30_000:
        verdict = "TINY — paste the whole repo, skip retrieval entirely"
    elif total_tokens < 100_000:
        verdict = "SMALL — fits in one context, repo map is plenty"
    elif total_tokens < 500_000:
        verdict = "MEDIUM — repo map + on-demand file reads"
    else:
        verdict = "LARGE — needs real retrieval infrastructure"
    print(f"  Verdict: {verdict}")
    print()

    # ─── By extension ──────────────────────────────────────────────
    print("─" * 64)
    print("  BY EXTENSION")
    print("─" * 64)
    by_ext = (
        df.groupby("ext")
          .agg(files=("path", "count"),
               lines=("lines", "sum"),
               tokens=("tokens", "sum"))
          .sort_values("tokens", ascending=False)
    )
    by_ext["% of repo"] = (by_ext["tokens"] / total_tokens * 100).round(1)
    print(by_ext.to_string())
    print()

    # ─── Top files ─────────────────────────────────────────────────
    print("─" * 64)
    print("  TOP 10 FILES BY TOKEN COUNT")
    print("─" * 64)
    top = df.nlargest(10, "tokens")[["path", "tokens", "lines"]].copy()
    top["% of repo"] = (top["tokens"] / total_tokens * 100).round(1)
    print(top.to_string(index=False))
    print()

    # ─── Distribution ──────────────────────────────────────────────
    print("─" * 64)
    print("  DISTRIBUTION")
    print("─" * 64)
    print(f"  Median file: {df['tokens'].median():>8,.0f} tokens")
    print(f"  Mean file:   {df['tokens'].mean():>8,.0f} tokens")
    print(f"  Largest:     {df['tokens'].max():>8,.0f} tokens")
    print(f"  Smallest:    {df['tokens'].min():>8,.0f} tokens")

    n_top = max(1, len(df) // 10)
    top_share = df.nlargest(n_top, "tokens")["tokens"].sum() / total_tokens * 100
    print(f"  Top 10% of files account for {top_share:.0f}% of all tokens")
    print("═" * 64)

════════════════════════════════════════════════════════════════
  REPO SIZE SUMMARY
════════════════════════════════════════════════════════════════
  Path:   /Users/robertandersson/dev/satellite-worker
  Files:          11
  Lines:      14,678
  Tokens:    124,749  (cl100k_base, ~equivalent to Claude)

  Verdict: MEDIUM — repo map + on-demand file reads

────────────────────────────────────────────────────────────────
  BY EXTENSION
────────────────────────────────────────────────────────────────
       files  lines  tokens  % of repo
ext                                   
.ts        5  14352  121792       97.6
.md        3    254    2463        2.0
.json      2     48     361        0.3
.yml       1     24     133        0.1

────────────────────────────────────────────────────────────────
  TOP 10 FILES BY TOKEN COUNT
────────────────────────────────────────────────────────────────
                                                                 path  tokens  lines  % of repo
/User